# ML Results Review

Review saved `run_ml_baselines.py` outputs without retraining models.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE = Path('..') if Path('../run_ml_baselines.py').exists() else Path('.')
RUN_DIR = BASE / 'artifacts' / 'baseline_v0.1'
METRICS = RUN_DIR / 'metrics.csv'
SUMMARY = RUN_DIR / 'metrics_summary.csv'
PREDICTION_DIR = RUN_DIR / 'predictions'
TRIALS = RUN_DIR / 'optuna_trials.csv'

print(RUN_DIR.resolve())


In [ ]:
metrics = pd.read_csv(METRICS) if METRICS.exists() else pd.DataFrame()
summary = pd.read_csv(SUMMARY) if SUMMARY.exists() else pd.DataFrame()
prediction_files = sorted(PREDICTION_DIR.glob('*_predictions.parquet')) if PREDICTION_DIR.exists() else []
if not prediction_files and PREDICTION_DIR.exists():
    prediction_files = sorted(PREDICTION_DIR.glob('*_predictions.csv'))
predictions = pd.read_parquet(prediction_files[0]) if prediction_files and prediction_files[0].suffix == '.parquet' else (pd.read_csv(prediction_files[0], parse_dates=['time_utc', 'target_time_utc']) if prediction_files else pd.DataFrame())
trials = pd.read_csv(TRIALS) if TRIALS.exists() else pd.DataFrame()
print('metrics', metrics.shape)
print('summary', summary.shape)
print('prediction files', len(prediction_files))
print('loaded predictions sample', predictions.shape)
print('trials', trials.shape)
summary.head(20)


In [ ]:
if not summary.empty:
    fig = px.bar(
        summary,
        x='model',
        y='rmse',
        color='horizon',
        facet_col='station',
        barmode='group',
        title='RMSE by station, horizon, and model',
        height=520,
    )
    fig.update_xaxes(tickangle=35)
    fig.show()


In [ ]:
station = predictions['station'].iloc[0] if not predictions.empty else None
horizon = predictions['horizon'].iloc[0] if not predictions.empty else None
model = predictions['model'].iloc[0] if not predictions.empty else None
print(station, horizon, model)
if station is not None:
    subset = predictions[(predictions['station'] == station) & (predictions['horizon'] == horizon) & (predictions['model'] == model)].copy()
    subset = subset.sort_values('target_time_utc').head(14 * 24 * 4)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=subset['target_time_utc'], y=subset['actual'], mode='lines', name='Actual foF2'))
    fig.add_trace(go.Scatter(x=subset['target_time_utc'], y=subset['predicted'], mode='lines', name='Predicted foF2'))
    fig.update_layout(title=f'{station} {horizon} {model}: actual vs predicted', xaxis_title='Target time, UTC', yaxis_title='foF2, MHz', template='plotly_white', height=520)
    fig.show()


In [ ]:
if not trials.empty:
    display(trials.sort_values('value').head(20))
